# Intro  
In this laboratory, you will learn more about clustering techniques. You will first implement your own
version of the K-means algorithm. Then you will apply it to different datasets and evaluate the performance
achieved. In the last part of the laboratory you will work on textual data, a domain where the data
preparation phase is crucial to any subsequent task. Specifically, you will try to detect topics out of a set
of real-world news data. Finally, you will describe each cluster through frequent itemset mining and word
clouds.

**Libraries**  
As you may have already understood, the Python language comes with many handy functions and third-
party libraries that you need to master to avoid boilerplate code. In many cases, you should leverage them
to focus on the analysis process rather than its implementation.  
That said, we listed a series of libraries you can make use of in this laboratory:

- NumPy  
- scikit-learn  
- Natural Language Toolkit  
- SciPy  

# Datasets  
For this lab, three different datasets will be used. Here, you will learn more about them and how to retrieve
them.  
The first two are synthetic datasets, i.e. they contain data that has been generated by hand to match a
specific scientific need. Synthetic data are often used to test machine learning algorithms under conditions
that are unlikely to occur with real-world data. Both datasets have been used in Fränti and Sieranoja 2018.
You can find them and many more (eventually, more complex) on their official web page.  
The third dataset instead is a real-world dataset containing textual news belonging to different topics.

# Synthetic 2-D Gaussian clusters
Think of this whole section like this:
- You have a 2D plane (an x–y plot).
- On this plane there are 5,000 points.
- Those points are not random chaos: they form 15 “blobs”.

> Those blobs = globular / Gaussian clusters.


**What is a “Gaussian / globular cluster” in plain words?**
Take one cluster:  
- It has a center (mean, written μ): “most points are around here”.
- It has a spread (standard deviation, written σ): “how far they typically wander from the center”.

If you look at the x-coordinate alone:
- Most x’s are near μₓ,
- Fewer x’s are far away,
- The histogram of x looks like the classic bell curve.
Same idea for y.  
So a “Gaussian cluster” just means:  
> “A bunch of points forming a roundish blob around a center, where points close to the center are common and far points are rare.”  

Nothing more magical than that.  

The normal distribution formula is provided just because the rat of the lab wants to show his little teeth and scare you off:
$N(x; \mu; \sigma) = \sqrt{\frac{1}{2\pi\sigma^2}} \exp\left(-\frac{1}{2\sigma^2}(x-\mu)^2\right)$

The idea is:
- Input: a value x
- Output: “how likely” x is, given:
    - μ = center
    - σ = spread

Key intuition:
- If x is close to μ → the exponent is close to 0 → big value → very likely.
- If x is far from μ → the exponent is a big negative number → tiny value → very unlikely.

So that formula is just the mathematical definition of a bell curve.

- Each row in the dataset: x, y = coordinates of one point.
- These 5,000 points are arranged into 15 bell-shaped blobs in 2D space.
- Each blob = one Gaussian / globular cluster.

> Download here the DS: https://raw.githubusercontent.com/dbdmg/data-science-lab/master/datasets/2D_gauss_clusters.txt

# Chameleon
This synthetic dataset was originally introduced in Karypis, Han, and Kumar 1999. It contains again two-dimensional data points distributed along interleaved clusters with different shapes.  

You can download it at: https://raw.githubusercontent.com/dbdmg/data-science-lab/master/datasets/chameleon_clusters.txt  

Each of the 8,000 rows contains the x and y coordinates of a single point. These points are grouped in the Euclidean space in 6 different clusters.

# 20 Newsgroups
The 20 Newsgroups dataset was originally collected in Lang 1995. It includes approximately 20,000 documents, partitioned across 20 different newsgroups, each corresponding to a different topic.  

For the sake of this laboratory, we chose T ≤ 20 topics and sampled uniformly only documents belonging to them. As a consequence, you have K ≤ 20,000 documents uniformly distributed across I different bics.  

You can download the dataset at: https://github.com/dbdmg/data-science-lab/blob/master/datasets/T-newsgroups.zip?raw=true  

Each document is located in a different file, which contains the raw text of the news. The name of the file in an integer number and corresponds to its ID.

---

#### 3.1 K-means design and implementation
This exercise will focus on the K-means clustering technique. You will implement your own version of the algorithm and then you will test it on the two synthetic datasets.

> 1. Load the synthetic 2-D dataset containing Gaussian clusters.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../../Dataset/LAB8/2D_gauss_clusters.txt', sep = ',')
display(df)


#### 2. Plot the data points as a scatter chart using the Matplotlib library.
At first sight, you should see 15 different globular clusters. Given this distribution, which could be the most suitable clustering technique among the ones that you known? Why?

In [ ]:
plt.figure(figsize=(8,8))
sns.scatterplot(x=df['x'], y=df['y'])
plt.show()

I kinda see those 15 clusters.  
Just by looking at the points distribution, which clustering technique might be the best one?
- Only numbers.
    - good for: K-means / Ward’s hierarchical (after standardization), DBSCAN if low-dimensional
    - bad for: methods designed for purely categorical data (e.g. k-modes)
- Not many globular-ish clusters, but weird shapes:
    - good for: DBSCAN, single-link (or other density/graph-based methods)
    - bad for: K-means, Ward’s (they like round, blob-shaped clusters)
- I have 5000 points:
    - good for:  K-means or mini-batch K-means (scales well), DBSCAN if still low-dimensional
    - bad for: very heavy hierarchical on much larger n (here it’s OK but less scalable than K-means)
- I know k = 15:
    - good for: K-means / K-medoids or any algorithm that takes K as input
    - bad for: DBSCAN (no K), “pure” hierarchical if you don’t really care about the full tree
- It seems like there's a lot of noise and there are some outliers:
    - good for: DBSCAN / HDBSCAN, complete-link hierarchical, or k-means + explicit outlier handling
    - bad for: plain K-means alone, single-link (very sensitive to outliers)


> I would start with a KMeans and decide based on the results

# 3. Focus now on the K-means technique.
Later in this laboratory you will use the scikit-learn package. Many of its functionalities are exposed via an object-oriented interface. With this paradigm in mind, implement now the K-means algorithm and expose it as a Python class. Try to solve this exercise by using numpy APIs (☹️☹️☹️). The bare skeleton of your class should look like this:

```python
class KMeans:
    def __init_(self, nclusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self. centroids = None
        self.labels = None
    
    def fit-predict (self, X):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C).
        : return: labels : array, shape = N.
        """
        pass
```

The core method is fit_predict. It should execute the K-means with K=self.n_clusters finding the centroids and assigning the label to each data point. Note that the class is intended to be stateful: it must keep track of the obtained centroids and labels. The max_iter parameter should be used to specify how many iterations are allowed for the main loop of the algorithm.

KMeans:
- initialize K random centroids
- compute the distance between centroid - point, for each point
- assign that point to the closest centroid, assign the point to the corresponding cluster
- recompute the centroids: take all points assigned to a cluster, compute the mean = new centroid
- repeat until the distance between new centroid and old centroid is < threshold

In [ ]:
# the KMeans takes as input a matrix (5000, 2), it's basically our df but as an array

X = df.values
print(X)

In [ ]:
# K random centroids in data space
# I need to define the limits of the data space

df.describe()
# min	55608.000000	25631.000000
# max	983609.000000	984555.000000
# so use 


min_per_col = df.min()      # this returns the min per column --> I need overall min
abs_min = float('inf')
for m in min_per_col:
    if m < abs_min:
        abs_min = m
# print(abs_min)

# dumbass just do
abs_min = X.min().min()
print(abs_min)


# same for max

In [ ]:
n_clusters = 5
x_initial_centroids = np.random.randint(low=df.min().min(), high=df.max().max(), size=n_clusters)
y_initial_centroids = np.random.randint(low=df.min().min(), high=df.max().max(), size=n_clusters)

# print(x_initial_centroids)
# print(y_initial_centroids)

initial_centroids = []
for idx in range(len(x_initial_centroids)):
    initial_centroids_coordinates = np.append(x_initial_centroids[idx], y_initial_centroids[idx])
    initial_centroids.append(initial_centroids_coordinates)
initial_centroids = np.array(initial_centroids)
print(initial_centroids)

plt.figure(figsize=(8,8))
sns.scatterplot(x=X[:,0], y=X[:,1])
sns.scatterplot(x=initial_centroids[:,0], y=initial_centroids[:,1])
plt.show()

# to get good initial centroids you need to get lucky as fuck

#### I mean it works but ...
A really better idea is to randomly select n_clusters values from K, so that I'm not initializing centroids in a huge space window but I make one of the actual points a centroid so that I know (if the DS is normal) that it is likely close to other points!!

In [ ]:
# initialize centroids
n_clusters = 15
initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=n_clusters)       # take these rows
# np.random.choice(len(X), size=n_clusters, replace=False)
initial_centroids = X[initial_centroids_idx]

plt.figure(figsize=(8,8))
sns.scatterplot(x=X[:,0], y=X[:,1])
sns.scatterplot(x=initial_centroids[:,0], y=initial_centroids[:,1])
plt.show()


# WAAAAAAAAYYY better :)


In [ ]:
print(type(initial_centroids))
print(initial_centroids.shape)

print(type(X))
print(X.shape)

#### Occhio agli shape / matrix compatibility

In [ ]:
# just for matrix practice --> compatibility

print(X@initial_centroids.T)    # prodotto riga X (1 punto) * colonna (1 centroide)
# print(X*initial_centroids.T)  NO: the shape is correct for a matrix product, not a 'normal' product where you multiply each element at its position 
# print(X-initial_centroids.T)  NO: subtraction between matrixes == you need THE SAME EXACT SHAPE

# I hate Numpy
- X (shape (5000, 2))
- initial_centroids (shape (15, 2))

> At first glance, this looks incompatible, because none of the dimensions are 1. But broadcasting can still occur because NumPy tries to make both arrays compatible by automatically stretching the dimensions that are smaller.

1.	Array X (shape (5000, 2)): This is an array of 5000 points, where each point has two features (coordinates in 2D).  
2.	Array initial_centroids (shape (15, 2)): This is an array of 15 centroids, each with two features (coordinates in 2D).  

How does broadcasting work in this case?  
- NumPy stretches the second axis of initial_centroids (which has size 2) across the first axis of X (which has size 5000).
- NumPy aligns the two arrays on the second axis, meaning each point in X is broadcasted against each centroid in initial_centroids.

So instead of transposing, NumPy essentially expands the smaller array to match the larger array’s shape for the operation. In this case, it will operate between each point in X and all centroids in initial_centroids.  

Why broadcasting works in this case ("lucky"):  
- Both X and initial_centroids have 2 features (2D coordinates), so they can naturally broadcast against each other.
- The last dimension of X (shape (5000, 2)) aligns directly with the second dimension of initial_centroids (shape (15, 2)). This is why broadcasting can happen here without adding new dimensions.

What would happen if the dimensions were different:  
- If the feature dimensions of the points and the centroids were not the same (e.g., if X had 3 features and initial_centroids had 2 features), broadcasting wouldn’t work directly. In this case, you would need to adjust the dimensions to make the operation valid.


- If, for example, X has a shape of (5000, 3) and initial_centroids has a shape of (15, 2), you would have to expand the dimensions so that the broadcasting could take place.  
- You would add a new axis (dimension)

In [ ]:
# euclidian distance: sqr((x_i-x_j)^2 + (y_i-y_j)^2)
    # difference
    # ^2
    # sum
    # sqr



# with lists --> AVOID IT, very very slow:
ignore = False
if ignore:
    l1 = []
    for i in range(len(initial_centroids)):
        l2 = []
        for j in range(len(X)):
            difference = X[j] - initial_centroids[i]        #X[j]:[845753 636607] - initial_centroids[i]:[144335 229546] = distance: [701418 407061]
            sq_difference = difference**2
            sum_sq_difference = np.sum(sq_difference)       # drop the dimension on which you sum, becomes a single number
            distance = sum_sq_difference**0.5
            l2.append(distance)
        l1.append(l2)
    

# use arrays, way faster
# instead of l1 = [] --> initialize the array (2D matrix) you would like to fill --> 5000 rows, 15 columns


use_array = False
if use_array:
    from_to_matrix = np.zeros(shape=(len(X), len(initial_centroids)))

    for i in range(len(initial_centroids)):
        for j in range(len(X)):
            difference = X[j] - initial_centroids[i]
            sq_difference = difference**2
            sum_sq_difference = np.sum(sq_difference)
            distance = sum_sq_difference**0.5
            from_to_matrix[j][i] = distance

    print(from_to_matrix[:5][:])
    # will print the first 5 rows and all columns:
    # ex. [[810978.34030571 501969.92058489 344484.57839793 358979.68194593
    #   362067.70328904 197055.04218365 639423.12537552 216028.49761085
    #   232982.3035211   91925.58530137 417938.97825998 513824.61124882
    #   593245.78723241 337609.50366511 563511.54691719]
    #  [...]]
    # --> distances from first point X[0] (or the first row of X) to all 15 centroids, so it returns a vector with 15 values


# !!!!!!!!!!!!!
# use arrays BUT AVOID FOR LOOPS --> broadcasting = add third dimension on which to broadcast
# I want to do initial centroids (15, 2) - X (5000, 2) --> dimension incompatibility --> add 3rd dimension ... where?
# I want (15, 2) - (5000, 2) --> (5000, 15) from-to-matrix, so let's add dimensions so that I would have 5000 as shape[0] and 15 as shape[1]
# SO (15, 2) let's add a dimension here (1, 15, 2) so that when you perform operations between (1, 15, 2) and (5000, 2) the first dimension broadcasts to 5000
# While for (5000, 2) to make operations work I need to add a dimensions here (5000, 1, 2) so that when you perform operations between (1, 15, 2) and (5000, 1, 2) there aren't any compatibility problems: 1vs5000 ok ; 15vs1 ok ; 2vs2 ok
# !!!!!!!!!!!!!

broadcasting = False
if broadcasting:
    X_expanded = X.reshape(len(X), 1, 2)                                                #(5000, 1, 2)
    centroids_expanded = initial_centroids.reshape(1, len(initial_centroids), 2)        # (1, 15, 2)
    difference = X_expanded - centroids_expanded
    # print(difference.shape)   (5000, 15, 2)   --> I want shape (5000, 15) --> sum over the axis you want to drop: axis = 2
    sq_difference = difference**2
    sum_sq_difference = np.sum(sq_difference, axis=2)
    distance = sum_sq_difference**0.5
    print(distance[:6][:])


# OR use a df and then convert it into an array
# (I used one of the ways, but all the previous ways where I used arrays could be done with df and then converted
# --> pretty dumb but I like df):
from_df_to_array = False
if from_df_to_array:
    from_to_matrix_df = pd.DataFrame()

    for i in range(len(initial_centroids)):
        distances = []

        for j in range(len(X)):
            difference = X[j] - initial_centroids[i]
            sq_difference = difference**2
            sum_sq_difference = np.sum(sq_difference)
            distance = sum_sq_difference**0.5
            distances.append(distance)
            
        from_to_matrix_df[f'centroid_{i}']=distances
    display(from_to_matrix_df)
    from_to_matrix = pd.DataFrame.to_numpy(from_to_matrix_df)
    print(from_to_matrix.shape)

In [ ]:
# clean it up
X_expanded = X.reshape(len(X), 1, 2)                                                #(5000, 1, 2)
centroids_expanded = initial_centroids.reshape(1, len(initial_centroids), 2)        # (1, 15, 2)

difference = X_expanded - centroids_expanded
sq_difference = difference**2
sum_sq_difference = np.sum(sq_difference, axis=2)
distance = sum_sq_difference**0.5

print(distance[:6][:])

#### Nice now I have a cool 2D from-to matrix with all distances ... now?
> Like how do I tell my model that the points closer to a centroid belongs to his cluster ... also how many points belongs to that cluster:  

for each point (each row), check the distance matrix to all other centroids (each column) and pick the smallest, assign that point to that centroid = cluster

In [ ]:
cluster_labels = []
for row in distance:
    # pick smallest values from row, PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
    row_cluster = row.argmin()
    cluster_labels.append(row_cluster)

print(len(cluster_labels))

In [ ]:
# this is the scatter plot after the first iteration of the home-made KMeans

plt.figure(figsize=(8,8))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=cluster_labels)
plt.show()

# Jesus christ that's BAD

In [ ]:
print(initial_centroids.shape)
print(X.shape)

In [ ]:
distance_matrix_df = pd.DataFrame(distance)
distance_matrix_df['cluster'] = cluster_labels

grouped = distance_matrix_df.groupby('cluster')
for id, values in grouped:
    print(f"cluster: {id}")
    display(values)
    print()
    # print(values[id].sum())
    # print(len(values[id]))
    # print(values[id].sum() / len(values[id]))
    print(values[id].mean())
    break

# cool, I know that the new centorid should have this distance ... but it's a single value, these are not the coordinates of the new centroid!!!
# In fact you need to get the index of all the values belonging to the same cluster, use that index to get back these points coordinates
# then compute the mean on the X coordinates and the mean on the Y coordinates = new centroid!!

In [ ]:
distance_matrix_df = pd.DataFrame(distance)
distance_matrix_df['cluster'] = cluster_labels

new_centroids=[]
grouped = distance_matrix_df.groupby('cluster')
for id, values in grouped:
    
    # print(f"cluster: {id}")
    # display(values)
    # print()

    same_cluster_index = values.index
    same_cluster_coordinates = X[same_cluster_index]
    
    # the cluster could be empty --> the new centroid is just a random point :)
    # actually meh: if you really want to handle truly empty clusters,
    # you need to loop over all possible cluster IDs, because if a cluster is empty then it's not returned by the groupby :(
    # I don't know how == ignore :)

    if len(same_cluster_coordinates) == 0:
        new_centroid_idx = np.random.randint(low = 0, high=len(X))
        new_centroid = X[new_centroid_idx]
        print(f'DAAAAAAAAMMMMNNNN, got unlucky, no points are close to cluster {id}')
        print("I'll just assign a random value, best of luck man :)")
    else:
        x_coordinates_same_cluster = same_cluster_coordinates[:,0]
        y_coordinates_same_cluster = same_cluster_coordinates[:,1]
        new_centroid_x = x_coordinates_same_cluster.mean()
        new_centroid_y = y_coordinates_same_cluster.mean()
        new_centroid = [new_centroid_x, new_centroid_y]

    # print(new_centroid_x, new_centroid_y)
    new_centroids.append(new_centroid)

new_centroids = np.array(new_centroids)




# OR without grouping but masking (same) --> meh
option_2 = False
if option_2:
    new_centroids = []
    unique_clusters = np.unique(cluster_labels)
    for unique_cluster in unique_clusters:
        mask = distance_matrix_df['cluster'] == unique_cluster
        same_cluster_idx = distance_matrix_df[mask].index
        
        # if no values belong to that cluster --> throw a new random centroid in there
        if len(X[same_cluster_idx]) == 0:
            new_centroid_idx = np.random.randint(low = 0, high=len(X))
            new_centroid = X[new_centroid_idx]
            print(f'DAAAAAAAAMMMMNNNN, got unlucky, no points are close to cluster {unique_cluster}')
            print("I'll just assign a random value, best of luck man :)")
            
        else:
            new_centroid = (X[same_cluster_idx][:,0].mean() , X[same_cluster_idx][:,1].mean())
        
        new_centroids.append(new_centroid)

    new_centroids = np.array(new_centroids)
    print(new_centroids)


In [ ]:
class KMeans:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    




    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do thre mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()


if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/2D_gauss_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=True)

    final_plot(X, labels, centroids)